# Memory

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain.memory import ChatMessageHistory

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain_core.chat_history import InMemoryChatMessageHistory # Its same as ChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

# `ConversationChain` and `ConversationBufferMemory` is old fashioned now.
# USE RunnableWithMessageHistory, which is new Langchain standard
from langchain.chains import ConversationChain, ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory


In [2]:
# Model configuration
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
)

## Chat message history

In [ ]:
chat = openai_llm # Set up the language model to use for chat interactions

history = ChatMessageHistory() # Create a new conversation history object - This will store the back-and-forth messages in the conversation
history.add_ai_message("hi!") # Add AIMessage to the history - This represents a message sent by the AI assistant
history.add_user_message("what is the capital of France?") # Add a user's question to the conversation history - This represents a message sent by the user

In [18]:
history.messages

[AIMessage(content='hi!'),
 HumanMessage(content='what is the capital of France?')]

In [19]:
ai_response = chat.invoke(history.messages)
ai_response

AIMessage(content='The capital of France is Paris.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 20, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None}, id='run-5ab4cadc-cfa3-4571-a942-b093c9e1828b-0', usage_metadata={'input_tokens': 20, 'output_tokens': 7, 'total_tokens': 27})

In [20]:
history.add_ai_message(ai_response)
history.messages

[AIMessage(content='hi!'),
 HumanMessage(content='what is the capital of France?'),
 AIMessage(content='The capital of France is Paris.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 20, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None}, id='run-5ab4cadc-cfa3-4571-a942-b093c9e1828b-0', usage_metadata={'input_tokens': 20, 'output_tokens': 7, 'total_tokens': 27})]

In [ ]:
chat = openai_llm # Set up the language model to use for chat interactions

history1 = InMemoryChatMessageHistory() # Create a new conversation history object - This will store the back-and-forth messages in the conversation
history1.add_ai_message("hi!") # Add AIMessage to the history - This represents a message sent by the AI assistant
history1.add_user_message("what is the capital of France?") # Add a user's question to the conversation history - This represents a message sent by the user

In [ ]:
history1.messages

[AIMessage(content='hi!'),
 HumanMessage(content='what is the capital of France?')]

## RunnableWithMessageHistory - ConversationChain

`ConversationChain` and `ConversationBufferMemory` is old fashioned now.

USING **RunnableWithMessageHistory**, which is new Langchain standard

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | openai_llm # Create the runnable chain


In [ ]:
# Session-based message history store

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory() # Way to pass/prevent history
        # store[session_id] = ChatMessageHistory() # Another way to pass/prevent history
        # store[session_id] = history # Passing old history - Way to pass old history
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


In [28]:
response = conversation.invoke(
    {"input": "Hello!"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response)

content='Hello! How can I assist you today?' response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 94, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None} id='run-1157b267-7398-4c5a-a0a2-c1658baf81b0-0' usage_metadata={'input_tokens': 94, 'output_tokens': 9, 'total_tokens': 103}


In [23]:
response = conversation.invoke(
    {"input": "I am a MERN Stack developer"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response.content)

That's great! As a MERN Stack developer, you work with MongoDB, Express.js, React.js, and Node.js to build full-stack web applications. How can I assist you today? Are you working on a specific project or looking for help with something related to the MERN stack?


In [24]:
# Continue the same conversation
response = conversation.invoke(
    {"input": "Do you remember what I just said?"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response.content)

Yes, you mentioned that you are a MERN Stack developer. How can I assist you further?
